# Digital Product Funnel & Retention Analytics
### Beginner-friendly B.Tech CSE 1st-year project

**Project aim:** Understand how users move through a digital product, where they stop, and whether they return.

This notebook uses a **synthetic (practice) dataset**. It does not contain real customer information.

**Tools:** Python, Pandas, NumPy and Matplotlib.

**Main tasks from the project brief**
1. Define one row as one event and remove duplicate events.
2. Check timestamps, event order, user IDs and session IDs.
3. Build a conversion funnel and calculate conversion rates.
4. Use NumPy bootstrapping to estimate a confidence interval.
5. Calculate cohort retention and repeat engagement.
6. Compare results by device and acquisition source.
7. Test different session windows and funnel definitions.
8. Suggest experiments, while not claiming that observational data proves cause and effect.

## 1. Import the libraries

A library is a collection of ready-made Python tools. Pandas helps us work with tables, NumPy helps with calculations, and Matplotlib helps us make charts.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Make the random dataset reproducible
np.random.seed(42)

print("Libraries imported successfully!")

## 2. Create a practice dataset

Each row represents **one event** performed by a user. The columns match the project brief:
- `user_id`: identifies a user
- `session_id`: identifies a visit/session
- `event_name`: what happened
- `timestamp`: when it happened
- `device`: device used
- `source`: where the user came from

We create 500 users. A user may visit more than once. The data is generated for learning, so the results are examples and not real business findings.

In [ ]:
number_of_users = 500
devices = ["Mobile", "Desktop", "Tablet"]
sources = ["Social Media", "Search", "Advertisement", "Direct"]

event_rows = []
start_date = pd.Timestamp("2026-01-01")

for user_number in range(1, number_of_users + 1):
    user_id = f"U{user_number:04d}"
    device = np.random.choice(devices, p=[0.65, 0.28, 0.07])
    source = np.random.choice(sources, p=[0.35, 0.30, 0.20, 0.15])

    # Some users return for a second or third session
    session_count = np.random.choice([1, 2, 3], p=[0.55, 0.30, 0.15])

    for session_number in range(1, session_count + 1):
        session_id = f"{user_id}_S{session_number}"
        session_start = start_date + pd.Timedelta(
            days=int(np.random.randint(0, 90)),
            hours=int(np.random.randint(0, 24))
        )

        # Every session starts with a page view
        session_events = ["page_view"]

        if np.random.rand() < 0.72:
            session_events.append("signup")
        if "signup" in session_events and np.random.rand() < 0.78:
            session_events.append("product_view")
        if "product_view" in session_events and np.random.rand() < 0.55:
            session_events.append("add_to_cart")
        if "add_to_cart" in session_events and np.random.rand() < 0.65:
            session_events.append("checkout")
        if "checkout" in session_events and np.random.rand() < 0.60:
            session_events.append("purchase")

        # Add an engagement event for some returning sessions
        if session_number > 1 and np.random.rand() < 0.70:
            session_events.append("app_open")

        for event_position, event_name in enumerate(session_events):
            event_time = session_start + pd.Timedelta(minutes=event_position * 3)
            event_rows.append({
                "user_id": user_id,
                "session_id": session_id,
                "event_name": event_name,
                "timestamp": event_time,
                "device": device,
                "source": source
            })

df = pd.DataFrame(event_rows)
df = df.sort_values(["user_id", "timestamp"]).reset_index(drop=True)

print("Dataset created!")
print("Rows:", len(df))
print("Unique users:", df["user_id"].nunique())
df.head(10)

## 3. Save and understand the dataset

The CSV file is a simple table file. You can download it from Colab after running this cell.

In [ ]:
df.to_csv("product_data.csv", index=False)
print("Saved as product_data.csv")
print(df.shape)
print(df.info())

## 4. Data cleaning

Cleaning means checking the data and fixing common problems before analysis.

- **Event grain:** one row = one event.
- **Duplicate events:** repeated identical rows should be removed.
- **Missing values:** rows without a user, session, event or timestamp cannot be used reliably.
- **Timestamp:** convert it into a date-time format.
- **Ordering:** sort events by user and time.

In [ ]:
print("Missing values before cleaning:")
print(df.isnull().sum())

print("Duplicate rows before cleaning:", df.duplicated().sum())

clean_df = df.drop_duplicates().copy()

required_columns = ["user_id", "session_id", "event_name", "timestamp"]
clean_df = clean_df.dropna(subset=required_columns)

clean_df["timestamp"] = pd.to_datetime(clean_df["timestamp"])
clean_df = clean_df.sort_values(
    ["user_id", "session_id", "timestamp"]
).reset_index(drop=True)

print("Rows before cleaning:", len(df))
print("Rows after cleaning:", len(clean_df))
print("Duplicate rows after cleaning:", clean_df.duplicated().sum())

## 5. Validate users, sessions and event order

**Identity stitching** means connecting events that belong to the same known user. In this practice dataset, `user_id` is already available, so we use it to group events. Real systems may need to connect anonymous device IDs to logged-in user IDs; this dataset does not contain those anonymous IDs.

A session is a group of events associated with one visit. Here we use the supplied `session_id`. We check whether timestamps move forward within each session.

In [ ]:
# Check whether any session contains more than one user
users_per_session = clean_df.groupby("session_id")["user_id"].nunique()
print("Sessions linked to more than one user:", (users_per_session > 1).sum())

# Check event ordering inside each session
clean_df["previous_time"] = clean_df.groupby("session_id")["timestamp"].shift(1)
clean_df["time_is_out_of_order"] = (
    clean_df["timestamp"] < clean_df["previous_time"]
)
print("Out-of-order events:", clean_df["time_is_out_of_order"].sum())

# Show the order of events in one example session
example_session = clean_df["session_id"].iloc[0]
print("Example session:", example_session)
print(
    clean_df[clean_df["session_id"] == example_session][
        ["event_name", "timestamp"]
    ]
)

## 6. Build the conversion funnel

A funnel shows how many unique users reach each step. We count a user once at each stage, even if the user creates repeated events.

For this project, the stages are:
1. Page view
2. Signup
3. Product view
4. Add to cart
5. Checkout
6. Purchase

Conversion rate at a stage = users at this stage ÷ users at the first stage × 100.

In [ ]:
funnel_events = [
    "page_view", "signup", "product_view",
    "add_to_cart", "checkout", "purchase"
]

users_at_stage = []
for event in funnel_events:
    users_at_stage.append(
        clean_df.loc[clean_df["event_name"] == event, "user_id"].nunique()
    )

funnel = pd.DataFrame({
    "stage": funnel_events,
    "users": users_at_stage
})

first_stage_users = funnel.loc[0, "users"]
funnel["conversion_from_start_percent"] = (
    funnel["users"] / first_stage_users * 100
).round(2)

funnel["conversion_from_previous_percent"] = (
    funnel["users"] / funnel["users"].shift(1) * 100
).round(2)
funnel.loc[0, "conversion_from_previous_percent"] = 100

display(funnel)

plt.figure(figsize=(9, 5))
plt.bar(funnel["stage"], funnel["users"])
plt.title("Digital Product Conversion Funnel")
plt.xlabel("Funnel stage")
plt.ylabel("Unique users")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 7. Bootstrap confidence interval using NumPy

A conversion percentage from a sample can change if we had a different sample of users. Bootstrapping repeatedly samples users with replacement and calculates the conversion rate each time.

We use 1,000 repeats and show an approximate 95% confidence interval. This is an estimate, not a guarantee.

In [ ]:
user_table = clean_df.groupby("user_id")["event_name"].apply(list).reset_index()
user_table["converted"] = user_table["event_name"].apply(
    lambda events: int("purchase" in events)
)

conversion_values = user_table["converted"].to_numpy()
bootstrap_rates = []

for repeat in range(1000):
    sample = np.random.choice(
        conversion_values,
        size=len(conversion_values),
        replace=True
    )
    bootstrap_rates.append(sample.mean() * 100)

lower_limit = np.percentile(bootstrap_rates, 2.5)
upper_limit = np.percentile(bootstrap_rates, 97.5)
actual_rate = conversion_values.mean() * 100

print(f"Purchase conversion rate: {actual_rate:.2f}%")
print(f"Approximate 95% bootstrap interval: {lower_limit:.2f}% to {upper_limit:.2f}%")

## 8. Cohort retention and repeat engagement

A **cohort** is a group of users who started in the same time period. Here, a user's cohort is the month of their first recorded event.

For a simple retention view, we check whether a user had any event in the month after their first month. This is monthly return activity, not proof that the user continuously used the product.

In [ ]:
first_activity = (
    clean_df.groupby("user_id")["timestamp"]
    .min()
    .reset_index(name="first_timestamp")
)
first_activity["cohort_month"] = (
    first_activity["first_timestamp"].dt.to_period("M")
)

activity = clean_df[["user_id", "timestamp"]].drop_duplicates().copy()
activity["activity_month"] = activity["timestamp"].dt.to_period("M")

cohort_activity = activity.merge(
    first_activity[["user_id", "cohort_month"]],
    on="user_id",
    how="left"
)
cohort_activity["months_after_start"] = (
    cohort_activity["activity_month"].astype("int64")
    - cohort_activity["cohort_month"].astype("int64")
)

cohort_counts = (
    cohort_activity.groupby(["cohort_month", "months_after_start"])["user_id"]
    .nunique()
    .reset_index(name="active_users")
)
cohort_sizes = (
    first_activity.groupby("cohort_month")["user_id"]
    .nunique()
    .reset_index(name="cohort_users")
)
cohort_table = cohort_counts.merge(cohort_sizes, on="cohort_month")
cohort_table["retention_percent"] = (
    cohort_table["active_users"] / cohort_table["cohort_users"] * 100
).round(1)

retention_pivot = cohort_table.pivot(
    index="cohort_month",
    columns="months_after_start",
    values="retention_percent"
)
display(retention_pivot.fillna(0))

# Repeat engagement: users with events in more than one session
sessions_per_user = clean_df.groupby("user_id")["session_id"].nunique()
repeat_users = (sessions_per_user > 1).sum()
repeat_percent = repeat_users / len(sessions_per_user) * 100

print("Users with more than one session:", repeat_users)
print(f"Repeat-session user percentage: {repeat_percent:.2f}%")

## 9. Compare funnel results by device and source

This is a segment comparison. It tells us whether the observed conversion rates differ between groups. It does **not** tell us that the device or source caused the difference.

In [ ]:
user_segments = (
    clean_df.groupby("user_id")
    .agg(
        device=("device", "first"),
        source=("source", "first"),
        purchased=("event_name", lambda x: int("purchase" in set(x)))
    )
    .reset_index()
)

device_summary = (
    user_segments.groupby("device")
    .agg(users=("user_id", "count"), purchasers=("purchased", "sum"))
    .reset_index()
)
device_summary["conversion_percent"] = (
    device_summary["purchasers"] / device_summary["users"] * 100
).round(2)

source_summary = (
    user_segments.groupby("source")
    .agg(users=("user_id", "count"), purchasers=("purchased", "sum"))
    .reset_index()
)
source_summary["conversion_percent"] = (
    source_summary["purchasers"] / source_summary["users"] * 100
).round(2)

print("Device comparison")
display(device_summary)
print("Acquisition source comparison")
display(source_summary)

## 10. Test different session windows

The supplied `session_id` is our main session definition. To see how a different rule could change the result, we also create a simple alternative session ID: a new session begins when the gap between two events for the same user is greater than 30 minutes or 60 minutes.

This is a sensitivity check. It does not replace the original session IDs.

In [ ]:
def count_sessions_with_window(data, minutes):
    ordered = data.sort_values(["user_id", "timestamp"]).copy()
    ordered["gap_minutes"] = (
        ordered.groupby("user_id")["timestamp"].diff().dt.total_seconds() / 60
    )
    ordered["new_session"] = (
        ordered["gap_minutes"].isna()
        | (ordered["gap_minutes"] > minutes)
    )
    ordered["alternative_session_number"] = (
        ordered.groupby("user_id")["new_session"].cumsum()
    )
    return ordered.groupby("user_id")["alternative_session_number"].nunique().sum()

session_comparison = pd.DataFrame({
    "session_rule": ["Provided session_id", "30-minute inactivity", "60-minute inactivity"],
    "total_sessions": [
        clean_df["session_id"].nunique(),
        count_sessions_with_window(clean_df, 30),
        count_sessions_with_window(clean_df, 60)
    ]
})
display(session_comparison)

## 11. Test an alternative funnel definition

The main funnel requires users to reach each named event. An alternative, simpler funnel can compare only page viewers with purchasers. We calculate both definitions so we can see that the answer depends on how a funnel is defined.

In [ ]:
main_funnel_conversion = (
    clean_df.loc[clean_df["event_name"] == "purchase", "user_id"].nunique()
    / clean_df.loc[clean_df["event_name"] == "page_view", "user_id"].nunique()
    * 100
)

page_to_purchase_users = clean_df.groupby("user_id")["event_name"].apply(
    lambda events: ("page_view" in set(events)) and ("purchase" in set(events))
)
simple_conversion = page_to_purchase_users.mean() * 100

print(f"Main funnel page-view-to-purchase conversion: {main_funnel_conversion:.2f}%")
print(f"Alternative user-level page-view-and-purchase conversion: {simple_conversion:.2f}%")

## 12. Findings, experiment ideas and limitations

Use the actual tables printed above when writing your final report. Do not copy made-up conclusions.

**Possible experiment hypotheses**
1. If the product page explains the product more clearly, then more users may add an item to the cart.
2. If the checkout form is shorter, then more users who start checkout may complete a purchase.
3. If the mobile layout is easier to use, then mobile users may complete the funnel more often.

**How to test:** Randomly assign eligible users to a control version and a changed version, keep other conditions similar, choose a success metric before starting, and compare the groups after enough data has been collected.

**Important difference**
- Observational finding: "In this generated dataset, one device group has a higher observed conversion rate."
- Causal result: "The new design increased conversion." This claim needs a well-designed experiment; a segment comparison alone is not enough.

**Limitations**
- This is synthetic practice data, not real product data.
- The user ID is already known; real identity stitching can be more complicated.
- The event sequence is simplified.
- The confidence interval depends on the sampling method and dataset.
- Retention depends on the chosen time period and definition.

## 13. Export the final project files

Run this cell at the end to save the cleaned events and analysis tables as CSV files. In Colab, use the Files panel on the left to download them.

In [ ]:
clean_df.drop(columns=["previous_time", "time_is_out_of_order"], errors="ignore").to_csv(
    "cleaned_events.csv", index=False
)
funnel.to_csv("funnel_table.csv", index=False)
cohort_table.to_csv("cohort_retention_table.csv", index=False)
device_summary.to_csv("device_comparison.csv", index=False)
source_summary.to_csv("source_comparison.csv", index=False)

print("Project output CSV files saved!")